# Phase 3: Baselines

In [1]:
import sys
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "google.colab" in sys.modules:
    if "/content" not in sys.path:
        sys.path.insert(0, "/content")
else:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from project_paths import *

In [2]:
def load_npz(path):
    data = np.load(path)
    return data['X'], data['Y']

x_train, y_train = load_npz(PROCESSED_DATA_DIR / "train" / "sequences.npz")
x_val, y_val = load_npz(PROCESSED_DATA_DIR / "validation" / "sequences.npz")
x_test, y_test = load_npz(PROCESSED_DATA_DIR / "test" / "sequences.npz")

print(f"Train shape: {x_train.shape} -> {y_train.shape}")
print(f"Val shape: {x_val.shape} -> {y_val.shape}")
print(f"Test shape: {x_test.shape} -> {y_test.shape}")

history_steps = x_test.shape[1]
horizon_steps = y_test.shape[1]
nodes = x_test.shape[2]
features = x_test.shape[3]

print(f"\nExtracted Config:")
print(f"History: {history_steps}")
print(f"Horizon: {horizon_steps}")
print(f"Nodes: {nodes}")
print(f"Features: {features}")

Train shape: (23976, 12, 207, 1) -> (23976, 3, 207, 1)
Val shape: (5126, 12, 207, 1) -> (5126, 3, 207, 1)
Test shape: (5128, 12, 207, 1) -> (5128, 3, 207, 1)

Extracted Config:
History: 12
Horizon: 3
Nodes: 207
Features: 1


In [3]:
from src.data.dataset import get_dataloaders

batch_size = 64
train_loader, val_loader, test_loader = get_dataloaders(
    x_train, y_train,
    x_val, y_val,
    x_test, y_test,
    batch_size=batch_size
)

batch_x, batch_y = next(iter(test_loader))
print(f"Batch X: {batch_x.shape}")
print(f"Batch Y: {batch_y.shape}")

Batch X: torch.Size([64, 12, 207, 1])
Batch Y: torch.Size([64, 3, 207, 1])


## Instantiate Models & Evaluate

In [4]:
from src.models.baseline import HistoricalAverage, LastValue, LinearRegressionModel, LSTMModel
from src.evaluation.evaluate import evaluate_model_comprehensive
from src.training.train import train_model
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
ha_model = HistoricalAverage(horizon_steps=horizon_steps)
lv_model = LastValue(horizon_steps=horizon_steps)

ha_results = evaluate_model_comprehensive(ha_model, test_loader, device)
lv_results = evaluate_model_comprehensive(lv_model, test_loader, device)

print("HA Metrics:", ha_results['overall'])
print("LV Metrics:", lv_results['overall'])

HA Metrics: {'MAE': 0.26324811577796936, 'RMSE': 0.5938243269920349, 'MAPE': 0.2452172040939331, 'WAPE': 0.3180343210697174, 'R2': 0.7585303783416748}
LV Metrics: {'MAE': 0.18541143834590912, 'RMSE': 0.44884636998176575, 'MAPE': 0.19639407098293304, 'WAPE': 0.223998561501503, 'R2': 0.862043559551239}


In [6]:
lr_model = LinearRegressionModel(history_steps, horizon_steps, nodes, features)
lstm_model = LSTMModel(history_steps, horizon_steps, nodes, features, hidden_dim=64, num_layers=2)

lr_optimizer = torch.optim.Adam(lr_model.parameters(), lr=0.001)
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
loss_fn = nn.L1Loss()

print("Training LR...")
train_model(lr_model, train_loader, val_loader, lr_optimizer, loss_fn, num_epochs=3, device=device)
print("Training LSTM...")
train_model(lstm_model, train_loader, val_loader, lstm_optimizer, loss_fn, num_epochs=3, device=device)

Training LR...


Epoch 1/3 - Train Loss: 0.3134 - Val Loss: 0.2967 - Val MAE: 0.2967


Epoch 2/3 - Train Loss: 0.2712 - Val Loss: 0.2906 - Val MAE: 0.2906


Epoch 3/3 - Train Loss: 0.2561 - Val Loss: 0.2687 - Val MAE: 0.2687
Training LSTM...


Epoch 1/3 - Train Loss: 0.2374 - Val Loss: 0.1882 - Val MAE: 0.1882


Epoch 2/3 - Train Loss: 0.1783 - Val Loss: 0.1853 - Val MAE: 0.1853


Epoch 3/3 - Train Loss: 0.1767 - Val Loss: 0.1810 - Val MAE: 0.1810


{'train_loss': [0.23743482088700588, 0.17829431921074618, 0.1767262743837562],
 'val_loss': [0.18821875146948822, 0.18532134185556412, 0.18095801761675476],
 'val_mae': [0.18821875146948822, 0.18532134185556412, 0.18095801761675476]}

In [7]:
lr_results = evaluate_model_comprehensive(lr_model, test_loader, device)
lstm_results = evaluate_model_comprehensive(lstm_model, test_loader, device)

print("LR Metrics:", lr_results['overall'])
print("LSTM Metrics:", lstm_results['overall'])

LR Metrics: {'MAE': 0.30180782079696655, 'RMSE': 0.531749963760376, 'MAPE': 0.24652624130249023, 'WAPE': 0.3646189272403717, 'R2': 0.8063749670982361}
LSTM Metrics: {'MAE': 0.1732899397611618, 'RMSE': 0.43329453468322754, 'MAPE': 0.13483339548110962, 'WAPE': 0.20935438573360443, 'R2': 0.8714379072189331}


In [8]:
results_dir = PROJECT_ROOT / 'experiments' / 'baseline'
results_dir.mkdir(parents=True, exist_ok=True)

results = [
    {'model': 'HistoricalAverage', **ha_results['overall']},
    {'model': 'LastValue', **lv_results['overall']},
    {'model': 'LinearRegression', **lr_results['overall']},
    {'model': 'LSTM', **lstm_results['overall']}
]
results_df = pd.DataFrame(results)
results_df.to_csv(results_dir / 'results.csv', index=False)
print("Results saved successfully.")
results_df

Results saved successfully.


,model,MAE,RMSE,MAPE,WAPE,R2
0,HistoricalAverage,0.263248,0.593824,0.245217,0.318034,0.758530
1,LastValue,0.185411,0.448846,0.196394,0.223999,0.862044
2,LinearRegression,0.301808,0.531750,0.246526,0.364619,0.806375
3,LSTM,0.173290,0.433295,0.134833,0.209354,0.871438
